In [2]:
!pip install -q transformers datasets evaluate accelerate rouge_score sentencepiece pandas numpy

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00


In [3]:
import os
import json
import numpy as np
import pandas as pd

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
import evaluate

In [4]:
EXPERIMENT_NAME = "raw_distilbart"

TRAIN_PATH = "/content/train_raw.csv"
VAL_PATH = "/content/val_raw.csv"
TEST_PATH = "/content/test_raw.csv"

MODEL_NAME = "sshleifer/distilbart-cnn-12-6"
OUTPUT_DIR = f"/content/{EXPERIMENT_NAME}_results"

In [5]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())

Train shape: (253, 6)
Validation shape: (54, 6)
Test shape: (55, 6)


,doc_id,doc_name,chunk_id,input_text,target_summary,split
0,0,Privacy Policy,0,search encrypt does not track search history i...,this service does not track you.,train
1,1,Privacy Policy,0,we also provide you additional data control op...,you can request access and deletion of persona...,train
2,2,Additional Terms of Service,0,rvices you grant oath the following worldwide ...,the copyright license granted to yahoo for pho...,train
3,4,Privacy Policy,0,it also enables us to serve you advertising an...,the service uses your personal data to employ ...,train
4,6,Terms of Use,0,you are not permitted to attempt to overload f...,this service prohibits users sending chain let...,train


In [6]:
def find_first_existing(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

INPUT_CANDIDATES = ["input_text", "clean_text", "original_text", "text", "source_text"]
TARGET_CANDIDATES = ["target_summary", "clean_summary", "reference_summary", "case_text", "summary"]

input_col = find_first_existing(train_df, INPUT_CANDIDATES)
target_col = find_first_existing(train_df, TARGET_CANDIDATES)

print("Detected input column:", input_col)
print("Detected target column:", target_col)

if input_col is None or target_col is None:
    raise ValueError("Could not detect input or target summary columns.")

Detected input column: input_text
Detected target column: target_summary


In [7]:
train_df = train_df[[input_col, target_col]].dropna().reset_index(drop=True)
val_df = val_df[[input_col, target_col]].dropna().reset_index(drop=True)
test_df = test_df[[input_col, target_col]].dropna().reset_index(drop=True)

train_df = train_df.rename(columns={input_col: "input_text", target_col: "target_summary"})
val_df = val_df.rename(columns={input_col: "input_text", target_col: "target_summary"})
test_df = test_df.rename(columns={input_col: "input_text", target_col: "target_summary"})

print(train_df.shape, val_df.shape, test_df.shape)
display(train_df.head())

(253, 2) (54, 2) (55, 2)


,input_text,target_summary
0,search encrypt does not track search history i...,this service does not track you.
1,we also provide you additional data control op...,you can request access and deletion of persona...
2,rvices you grant oath the following worldwide ...,the copyright license granted to yahoo for pho...
3,it also enables us to serve you advertising an...,the service uses your personal data to employ ...
4,you are not permitted to attempt to overload f...,this service prohibits users sending chain let...


In [8]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

dataset_dict

DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_summary'],
        num_rows: 253
    })
    validation: Dataset({
        features: ['input_text', 'target_summary'],
        num_rows: 54
    })
    test: Dataset({
        features: ['input_text', 'target_summary'],
        num_rows: 55
    })
})

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print("Loaded model:", MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Please make sure the generation config includes `forced_bos_token_id=0`. 


model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loaded model: sshleifer/distilbart-cnn-12-6


In [10]:
MAX_INPUT_LEN = 768
MAX_TARGET_LEN = 64

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=MAX_INPUT_LEN,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["target_summary"],
        max_length=MAX_TARGET_LEN,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset_dict.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset_dict["train"].column_names
)

tokenized_datasets

Map:   0%|          | 0/253 [00:00<?, ? examples/s]

Map:   0%|          | 0/54 [00:00<?, ? examples/s]

Map:   0%|          | 0/55 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 253
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 54
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 55
    })
})

In [11]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [12]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels)

    return {
        "rouge1": round(result["rouge1"], 4),
        "rouge2": round(result["rouge2"], 4),
        "rougeL": round(result["rougeL"], 4),
        "rougeLsum": round(result["rougeLsum"], 4),
    }

In [13]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=2,
    predict_with_generate=True,
    logging_steps=20,
    fp16=False,
    report_to="none"
)

In [14]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [15]:
base_val_results = trainer.evaluate(tokenized_datasets["validation"])
print("Distilled base model validation results:", base_val_results)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Distilled base model validation results: {'eval_loss': 4.544017791748047, 'eval_model_preparation_time': 0.0256, 'eval_rouge1': 0.2061, 'eval_rouge2': 0.0876, 'eval_rougeL': 0.1734, 'eval_rougeLsum': 0.1737, 'eval_runtime': 434.5807, 'eval_samples_per_second': 0.124, 'eval_steps_per_second': 0.062}


In [16]:
base_test_results = trainer.evaluate(tokenized_datasets["test"])
print("Distilled base model test results:", base_test_results)

Distilled base model test results: {'eval_loss': 4.700263977050781, 'eval_model_preparation_time': 0.0256, 'eval_rouge1': 0.1868, 'eval_rouge2': 0.073, 'eval_rougeL': 0.1511, 'eval_rougeLsum': 0.1503, 'eval_runtime': 430.4941, 'eval_samples_per_second': 0.128, 'eval_steps_per_second': 0.065}


In [17]:
train_result = trainer.train()
print("Training finished.")
print(train_result.metrics)

Epoch,Training Loss,Validation Loss,Model Preparation Time,Rouge1,Rouge2,Rougel,Rougelsum
1,2.171362,1.866877,0.025600,0.236400,0.125000,0.201000,0.201500
2,1.217294,1.760521,0.025600,0.245900,0.135600,0.211000,0.211600


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training finished.
{'train_runtime': 3215.8125, 'train_samples_per_second': 0.157, 'train_steps_per_second': 0.079, 'total_flos': 85797259911168.0, 'train_loss': 1.9824439146387296, 'epoch': 2.0}


In [18]:
ft_val_results = trainer.evaluate(tokenized_datasets["validation"])
print("Fine-tuned distilled model validation results:", ft_val_results)

ft_test_results = trainer.evaluate(tokenized_datasets["test"])
print("Fine-tuned distilled model test results:", ft_test_results)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Fine-tuned distilled model validation results: {'eval_loss': 1.7605211734771729, 'eval_model_preparation_time': 0.0256, 'eval_rouge1': 0.2459, 'eval_rouge2': 0.1356, 'eval_rougeL': 0.211, 'eval_rougeLsum': 0.2116, 'eval_runtime': 372.3171, 'eval_samples_per_second': 0.145, 'eval_steps_per_second': 0.073, 'epoch': 2.0}
Fine-tuned distilled model test results: {'eval_loss': 2.1589930057525635, 'eval_model_preparation_time': 0.0256, 'eval_rouge1': 0.2254, 'eval_rouge2': 0.101, 'eval_rougeL': 0.1827, 'eval_rougeLsum': 0.1833, 'eval_runtime': 400.6269, 'eval_samples_per_second': 0.137, 'eval_steps_per_second': 0.07, 'epoch': 2.0}


In [19]:
sample_texts = test_df["input_text"].head(5).tolist()

inputs = tokenizer(
    sample_texts,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=MAX_INPUT_LEN
)

outputs = model.generate(
    **inputs,
    max_length=64,
    num_beams=4,
    length_penalty=1.0,
    early_stopping=True
)

generated_summaries = tokenizer.batch_decode(outputs, skip_special_tokens=True)

for i, summary in enumerate(generated_summaries):
    print("=" * 100)
    print("INPUT:")
    print(sample_texts[i][:700])
    print("\nGENERATED SUMMARY:")
    print(summary)
    print("\nREFERENCE SUMMARY:")
    print(test_df["target_summary"].iloc[i])

INPUT:
nothing in this agreement gives npm any ownership rights in intellectual property that you share with npm services such as your account information or any packages you share with npm services your content.

GENERATED SUMMARY:
this service assumes ownership of your data and ownership of any of your personal data that you share with the service that you send your personal information to the service and any of the packages you send you are involved in the service are not subject to ownership of intellectual property.

REFERENCE SUMMARY:
you maintain ownership of your data.
INPUT:
contests surveys and sweepstakes data this personal data is used to allow you to sign up and participate in these types of promotions. the exact personal data collected will vary depending on the promotion.

GENERATED SUMMARY:
the service uses your personal data for promotional purposes including those involved in the promotion. the service may use your personal information for other purposes including for